# 04 Model Train 06 – 차별화 포인트

## 차별화 포인트
- **구매 시점 예측**: 살지/안살지를 넘어 **언제 살지(구매 주기 버킷)**를 예측
- **공동 구매 패턴 (Lift)**: Aisle 단위 공동구매 **Lift = P(A∩B) / (P(A)×P(B))** 점수로
  함께 자주 사는 상품 연관성을 피처에 반영

### `05` 대비 개선점
| 항목 | 05 | 06 |
|------|----|----|  
| 공동구매 점수 | 정규화 공동구매 횟수 | **Lift** (통계적 연관성) |
| 타이밍 피처 | timing_ratio | timing_ratio + **hazard_proxy** (1회 구매 포함) |
| 모델 비교 | 없음 | **베이스라인 vs 차별화 비교** |

## 1. 데이터 로드

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from utils import reduce_memory_usage

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print('데이터 로드 중...')
orders       = pd.read_csv('../data/raw/orders.csv')
prior        = pd.read_csv('../data/raw/order_products__prior.csv')
train_actual = pd.read_csv('../data/raw/order_products__train.csv')
products     = pd.read_csv('../data/raw/products.csv')
aisles       = pd.read_csv('../data/raw/aisles.csv')

orders   = reduce_memory_usage(orders)
prior    = reduce_memory_usage(prior)
products = reduce_memory_usage(products)

print(f'orders  : {orders.shape}')
print(f'prior   : {prior.shape}')
print(f'train   : {train_actual.shape}')
print(f'products: {products.shape}')

## 2. 구매 시점 피처 생성 (언제 살지)

유저별 누적 경과일 → (user, product) 단위 구매 간격 → **timing_ratio** + **hazard_proxy** 계산.
`hazard_proxy`는 1회 구매 상품도 포함하기 위해 유저 평균 간격으로 대체한 overdue 비율입니다.

In [ ]:
print('유저별 누적 경과일 계산 중...')

prior_orders = orders[orders['eval_set'] == 'prior'].copy()
prior_orders = prior_orders.sort_values(['user_id', 'order_number'])
prior_orders['days_since_prior_order'] = prior_orders['days_since_prior_order'].fillna(0)
prior_orders['cum_days'] = (
    prior_orders.groupby('user_id')['days_since_prior_order'].cumsum()
)

prior_timed = prior.merge(
    prior_orders[['order_id', 'user_id', 'order_number', 'cum_days']],
    on='order_id', how='left'
)

print(f'prior_timed: {prior_timed.shape}')
print(prior_timed[['user_id', 'product_id', 'order_number', 'cum_days']].head())

In [ ]:
print('유저-상품별 구매 간격 통계 계산 중...')

pt = prior_timed.sort_values(['user_id', 'product_id', 'order_number']).reset_index(drop=True)
pt['prev_cum_days'] = pt.groupby(['user_id', 'product_id'])['cum_days'].shift(1)
pt['purchase_interval'] = pt['cum_days'] - pt['prev_cum_days']

up_timing = pt.groupby(['user_id', 'product_id']).agg(
    up_avg_interval   = ('purchase_interval', 'mean'),
    up_std_interval   = ('purchase_interval', 'std'),
    up_last_cum_days  = ('cum_days', 'max'),
    up_last_order_num = ('order_number', 'max'),
    up_buy_count      = ('order_id', 'count'),
).reset_index()

user_last_prior = (
    prior_orders.groupby('user_id')['cum_days'].max()
    .reset_index().rename(columns={'cum_days': 'user_last_cum'})
)
up_timing = up_timing.merge(user_last_prior, on='user_id', how='left')
up_timing['days_since_last_buy'] = up_timing['user_last_cum'] - up_timing['up_last_cum_days']
up_timing['up_avg_interval'] = up_timing['up_avg_interval'].fillna(-1)
up_timing['up_std_interval'] = up_timing['up_std_interval'].fillna(0)

del pt
print(f'up_timing: {up_timing.shape}')
print(up_timing.head().to_string(index=False))

In [ ]:
# timing_ratio: (마지막 구매 후 경과일 + train 주문까지 경과일) / 평균 구매 간격
# hazard_proxy: avg_interval 없는 1회 구매 상품도 유저 평균 간격으로 fallback
train_order_info = (
    orders[orders['eval_set'] == 'train'][['user_id', 'days_since_prior_order']]
    .rename(columns={'days_since_prior_order': 'train_days_since_prior'})
)
up_timing = up_timing.merge(train_order_info, on='user_id', how='left')

up_timing['total_gap'] = (
    up_timing['days_since_last_buy'] + up_timing['train_days_since_prior'].fillna(0)
)
up_timing['timing_ratio'] = np.where(
    up_timing['up_avg_interval'] > 0,
    up_timing['total_gap'] / up_timing['up_avg_interval'],
    0.0
)

user_avg = (
    up_timing[up_timing['up_avg_interval'] > 0]
    .groupby('user_id')['up_avg_interval'].mean()
    .reset_index().rename(columns={'up_avg_interval': 'user_avg_interval'})
)
up_timing = up_timing.merge(user_avg, on='user_id', how='left')
eff_iv = np.where(
    up_timing['up_avg_interval'] > 0,
    up_timing['up_avg_interval'],
    up_timing['user_avg_interval'].fillna(30)
)
up_timing['hazard_proxy'] = up_timing['total_gap'] / np.clip(eff_iv, 1, None)
up_timing.drop(columns=['total_gap', 'user_avg_interval'], inplace=True)

up_timing = reduce_memory_usage(up_timing)
print(f'타이밍 피처 완성: {up_timing.shape}')
print('\n[timing_ratio 분포 (2회 이상 구매)]')
print(up_timing[up_timing['up_avg_interval'] > 0]['timing_ratio'].describe().round(3))

## 3. 공동 구매 패턴 피처 생성 (Lift 기반)

단순 공동구매 횟수 대신 **Lift = P(A∩B) / (P(A)×P(B))** 를 사용합니다.  
Lift > 1이면 두 Aisle이 우연보다 더 자주 함께 구매됩니다.

In [ ]:
print('Aisle × Aisle Lift 행렬 계산 중...')

prod_aisle   = products[['product_id', 'aisle_id']].copy()
prior_aisled = prior.merge(prod_aisle, on='product_id', how='left')
order_aisle  = prior_aisled[['order_id', 'aisle_id']].drop_duplicates()

all_aisle_ids = sorted(order_aisle['aisle_id'].dropna().astype(int).unique())
all_order_ids = order_aisle['order_id'].unique()
order_to_idx  = {o: i for i, o in enumerate(all_order_ids)}
aisle_to_idx  = {a: i for i, a in enumerate(all_aisle_ids)}
n_orders_c    = len(all_order_ids)
n_aisles      = len(all_aisle_ids)

rows_m = order_aisle['order_id'].map(order_to_idx)
cols_m = order_aisle['aisle_id'].map(aisle_to_idx)
mask   = rows_m.notna() & cols_m.notna()

M = csr_matrix(
    (np.ones(mask.sum()), (rows_m[mask].astype(int), cols_m[mask].astype(int))),
    shape=(n_orders_c, n_aisles)
)

cooccur = (M.T @ M).toarray().astype(np.float32)
np.fill_diagonal(cooccur, 0)

# Lift = cooccur[i,j] * n_orders / (count[i] * count[j])
aisle_counts = np.array(M.sum(axis=0)).flatten().astype(np.float32)
outer        = np.outer(aisle_counts, aisle_counts)
lift_matrix  = np.divide(
    cooccur * n_orders_c, outer,
    where=outer > 0, out=np.zeros_like(cooccur)
)
np.fill_diagonal(lift_matrix, 0)

del cooccur, outer, prior_aisled, order_aisle
print(f'Lift 행렬: {lift_matrix.shape}  (Aisle 수: {n_aisles})')

aisle_name = dict(zip(aisles['aisle_id'], aisles['aisle']))
pairs = []
for i in range(n_aisles):
    j = int(np.argmax(lift_matrix[i]))
    if lift_matrix[i, j] > 1.0:
        pairs.append({
            'aisle_a': aisle_name.get(all_aisle_ids[i], all_aisle_ids[i]),
            'aisle_b': aisle_name.get(all_aisle_ids[j], all_aisle_ids[j]),
            'lift'   : round(float(lift_matrix[i, j]), 3)
        })
pairs_df = (
    pd.DataFrame(pairs)
    .sort_values('lift', ascending=False)
    .drop_duplicates().head(10)
)
print('\n[Lift 기준 상위 10개 공동 구매 Aisle 조합]')
print(pairs_df.to_string(index=False))

In [ ]:
print('유저-Aisle Lift 친화도 행렬 계산 중...')

_pa = prior.merge(orders[['order_id', 'user_id']], on='order_id', how='left')
_pa = _pa.merge(products[['product_id', 'aisle_id']], on='product_id', how='left')

user_aisle   = _pa.groupby(['user_id', 'aisle_id']).size().reset_index(name='buy_count')
del _pa

all_user_ids = sorted(user_aisle['user_id'].unique())
user_to_idx  = {u: i for i, u in enumerate(all_user_ids)}

rows_u = user_aisle['user_id'].map(user_to_idx)
cols_u = user_aisle['aisle_id'].map(aisle_to_idx)
data_u = user_aisle['buy_count'].values
mask_u = rows_u.notna() & cols_u.notna()

U = csr_matrix(
    (
        data_u[mask_u.values],
        (rows_u[mask_u].values.astype(int), cols_u[mask_u].values.astype(int))
    ),
    shape=(len(all_user_ids), n_aisles)
)

# U_lift[u, j] = 유저 u의 구매 이력 × lift_matrix → aisle j에 대한 Lift 가중 친화도
U_lift = (U @ lift_matrix).astype(np.float32)
del U

print(f'유저×Aisle Lift 친화도 행렬: {U_lift.shape}')
print('계산 완료')

## 4. 통합 피처 테이블 구성 (가드 블록 포함)

In [ ]:
import pandas as pd, numpy as np, sys
from scipy.sparse import csr_matrix
sys.path.append('..')

if 'reduce_memory_usage' not in vars():
    from utils import reduce_memory_usage

# 데이터 로드 가드
for _var, _path in [
    ('orders',   '../data/raw/orders.csv'),
    ('prior',    '../data/raw/order_products__prior.csv'),
    ('products', '../data/raw/products.csv'),
]:
    if _var not in vars():
        globals()[_var] = reduce_memory_usage(pd.read_csv(_path))
if 'aisles' not in vars():
    aisles = pd.read_csv('../data/raw/aisles.csv')

# up_timing 재계산 가드
if 'up_timing' not in vars():
    print('  → up_timing 재계산 중...')
    _po = orders[orders['eval_set'] == 'prior'].copy().sort_values(['user_id', 'order_number'])
    _po['days_since_prior_order'] = _po['days_since_prior_order'].fillna(0)
    _po['cum_days'] = _po.groupby('user_id')['days_since_prior_order'].cumsum()
    _pt = prior.merge(_po[['order_id', 'user_id', 'order_number', 'cum_days']], on='order_id', how='left')
    _pt = _pt.sort_values(['user_id', 'product_id', 'order_number']).reset_index(drop=True)
    _pt['prev_cum_days'] = _pt.groupby(['user_id', 'product_id'])['cum_days'].shift(1)
    _pt['purchase_interval'] = _pt['cum_days'] - _pt['prev_cum_days']
    up_timing = _pt.groupby(['user_id', 'product_id']).agg(
        up_avg_interval   = ('purchase_interval', 'mean'),
        up_std_interval   = ('purchase_interval', 'std'),
        up_last_cum_days  = ('cum_days', 'max'),
        up_last_order_num = ('order_number', 'max'),
        up_buy_count      = ('order_id', 'count'),
    ).reset_index()
    _ulp = _po.groupby('user_id')['cum_days'].max().reset_index().rename(columns={'cum_days': 'user_last_cum'})
    up_timing = up_timing.merge(_ulp, on='user_id', how='left')
    up_timing['days_since_last_buy'] = up_timing['user_last_cum'] - up_timing['up_last_cum_days']
    up_timing['up_avg_interval'] = up_timing['up_avg_interval'].fillna(-1)
    up_timing['up_std_interval'] = up_timing['up_std_interval'].fillna(0)
    _toi = (orders[orders['eval_set'] == 'train'][['user_id', 'days_since_prior_order']]
            .rename(columns={'days_since_prior_order': 'train_days_since_prior'}))
    up_timing = up_timing.merge(_toi, on='user_id', how='left')
    up_timing['total_gap'] = up_timing['days_since_last_buy'] + up_timing['train_days_since_prior'].fillna(0)
    up_timing['timing_ratio'] = np.where(up_timing['up_avg_interval'] > 0,
        up_timing['total_gap'] / up_timing['up_avg_interval'], 0.0)
    _ua = (up_timing[up_timing['up_avg_interval'] > 0]
           .groupby('user_id')['up_avg_interval'].mean()
           .reset_index().rename(columns={'up_avg_interval': 'user_avg_interval'}))
    up_timing = up_timing.merge(_ua, on='user_id', how='left')
    _eff = np.where(up_timing['up_avg_interval'] > 0,
                    up_timing['up_avg_interval'], up_timing['user_avg_interval'].fillna(30))
    up_timing['hazard_proxy'] = up_timing['total_gap'] / np.clip(_eff, 1, None)
    up_timing.drop(columns=['total_gap', 'user_avg_interval'], inplace=True)
    up_timing = reduce_memory_usage(up_timing)
    del _po, _pt, _ulp, _toi, _ua, _eff
    print('  → up_timing 완료')

# lift_matrix / user_to_idx / aisle_to_idx / U_lift 재계산 가드
if any(v not in vars() for v in ['lift_matrix', 'user_to_idx', 'aisle_to_idx', 'U_lift']):
    print('  → Lift 행렬 재계산 중...')
    _pa  = prior.merge(products[['product_id', 'aisle_id']], on='product_id', how='left')
    _oa  = _pa[['order_id', 'aisle_id']].drop_duplicates()
    all_aisle_ids = sorted(_oa['aisle_id'].dropna().astype(int).unique())
    _aoi = _oa['order_id'].unique()
    _o2i = {o: i for i, o in enumerate(_aoi)}
    aisle_to_idx  = {a: i for i, a in enumerate(all_aisle_ids)}
    _n_o = len(_aoi); _n_a = len(all_aisle_ids)
    _rm = _oa['order_id'].map(_o2i); _cm = _oa['aisle_id'].map(aisle_to_idx)
    _mk = _rm.notna() & _cm.notna()
    _M  = csr_matrix((np.ones(int(_mk.sum())),
                      (_rm[_mk].values.astype(int), _cm[_mk].values.astype(int))),
                     shape=(_n_o, _n_a))
    _co = (_M.T @ _M).toarray().astype(np.float32)
    np.fill_diagonal(_co, 0)
    _ac = np.array(_M.sum(axis=0)).flatten().astype(np.float32)
    _ot = np.outer(_ac, _ac)
    lift_matrix = np.divide(_co * _n_o, _ot, where=_ot > 0, out=np.zeros_like(_co))
    np.fill_diagonal(lift_matrix, 0)
    del _pa, _oa, _M, _co, _ac, _ot
    _pa2 = prior.merge(orders[['order_id', 'user_id']], on='order_id', how='left')
    _pa2 = _pa2.merge(products[['product_id', 'aisle_id']], on='product_id', how='left')
    user_aisle   = _pa2.groupby(['user_id', 'aisle_id']).size().reset_index(name='buy_count')
    del _pa2
    all_user_ids = sorted(user_aisle['user_id'].unique())
    user_to_idx  = {u: i for i, u in enumerate(all_user_ids)}
    _ru = user_aisle['user_id'].map(user_to_idx)
    _cu = user_aisle['aisle_id'].map(aisle_to_idx)
    _du = user_aisle['buy_count'].values
    _mu = _ru.notna() & _cu.notna()
    _U  = csr_matrix((_du[_mu.values],
                      (_ru[_mu].values.astype(int), _cu[_mu].values.astype(int))),
                     shape=(len(all_user_ids), _n_a))
    U_lift = (_U @ lift_matrix).astype(np.float32)
    del _ru, _cu, _du, _mu, _U
    print('  → Lift 행렬 완료')

# 기존 피처 테이블 로드
if 'data' not in vars():
    data = reduce_memory_usage(pd.read_csv('../data/prep/k-pick_total_v3.csv'))
    if 'label' in data.columns and 'reordered' not in data.columns:
        data = data.rename(columns={'label': 'reordered'})

# 타이밍 피처 결합
timing_cols = ['user_id', 'product_id', 'up_avg_interval', 'up_std_interval',
               'days_since_last_buy', 'timing_ratio', 'hazard_proxy', 'up_buy_count']
if 'timing_ratio' not in data.columns:
    data = data.merge(up_timing[timing_cols], on=['user_id', 'product_id'], how='left')

# 공동구매 Lift 점수 결합
if 'copurchase_lift' not in data.columns:
    if 'aisle_id' not in data.columns:
        data = data.merge(products[['product_id', 'aisle_id']], on='product_id', how='left')
    data['_uidx'] = data['user_id'].map(user_to_idx)
    data['_aidx'] = data['aisle_id'].map(aisle_to_idx)
    _v = data['_uidx'].notna() & data['_aidx'].notna()
    data['copurchase_lift'] = np.float32(0.0)
    data.loc[_v, 'copurchase_lift'] = U_lift[
        data.loc[_v, '_uidx'].astype(int).values,
        data.loc[_v, '_aidx'].astype(int).values
    ]
    data.drop(columns=['_uidx', '_aidx'], inplace=True)
    data = reduce_memory_usage(data)

print(f'통합 데이터: {data.shape}')
new_feats = ['up_avg_interval', 'up_std_interval', 'days_since_last_buy',
             'timing_ratio', 'hazard_proxy', 'up_buy_count', 'copurchase_lift']
print('\n[차별화 신규 피처 기술통계]')
print(data[new_feats].describe().round(3).to_string())

## 5. 모델 1 – 구매 시점 예측 (언제 살까?)

2회 이상 구매 이력이 있는 (user, product) 쌍에서 **4개 구매 주기 버킷**을 예측합니다.  
- 0: 매우 빈번 (≤7일) · 1: 빈번 (8-15일) · 2: 보통 (16-30일) · 3: 드물게 (>30일)

In [ ]:
timing_data = data[data['up_avg_interval'] > 0].copy()

timing_data['timing_label'] = pd.cut(
    timing_data['up_avg_interval'],
    bins=[0, 7, 15, 30, float('inf')],
    labels=[0, 1, 2, 3],
    right=True
).astype(int)

label_names = {0: '매우빈번(≤7일)', 1: '빈번(8-15일)', 2: '보통(16-30일)', 3: '드물게(>30일)'}
print('[구매 시점 버킷 분포]')
for k in range(4):
    cnt = (timing_data['timing_label'] == k).sum()
    print(f'  {k} {label_names[k]:15s}: {cnt:>10,}건  ({cnt/len(timing_data)*100:.1f}%)')

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split

unused_t = ['user_id', 'order_id', 'eval_set', 'product_id', 'reordered',
            'order_number', 'aisle_id', 'up_avg_interval', 'timing_label']
feat_cols_t = [c for c in timing_data.columns
               if c not in unused_t and timing_data[c].dtype != object]

X_t = timing_data[feat_cols_t].fillna(0)
y_t = timing_data['timing_label'].astype(int)

X_trv, X_te_t, y_trv, y_te_t = train_test_split(
    X_t, y_t, test_size=0.1, random_state=42, stratify=y_t)
X_tr_t, X_va_t, y_tr_t, y_va_t = train_test_split(
    X_trv, y_trv, test_size=2/9, random_state=42, stratify=y_trv)

print(f'[데이터 분리 (7:2:1)]  학습: {len(y_tr_t):,}  검증: {len(y_va_t):,}  테스트: {len(y_te_t):,}')

dtrain_t = lgb.Dataset(X_tr_t, label=y_tr_t)
dval_t   = lgb.Dataset(X_va_t, label=y_va_t, reference=dtrain_t)

params_t = {
    'objective'       : 'multiclass',
    'num_class'       : 4,
    'metric'          : 'multi_logloss',
    'boosting_type'   : 'gbdt',
    'learning_rate'   : 0.05,
    'num_leaves'      : 31,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq'    : 5,
    'seed'            : 42,
    'verbose'         : -1
}

print('\n[구매 시점 예측 모델 학습 시작]')
model_timing = lgb.train(
    params_t, dtrain_t, num_boost_round=500,
    valid_sets=[dval_t], valid_names=['valid'],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)
print(f'\n최적 반복 횟수: {model_timing.best_iteration}')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report

probs_t = model_timing.predict(X_te_t)
preds_t = probs_t.argmax(axis=1)

print('[구매 시점 예측 모델 성능]')
print(f'  Accuracy: {accuracy_score(y_te_t, preds_t):.4f}')
print()
print(classification_report(
    y_te_t, preds_t,
    target_names=['매우빈번(≤7일)', '빈번(8-15일)', '보통(16-30일)', '드물게(>30일)'],
    zero_division=0
))

imp_t = pd.DataFrame({
    'Feature': feat_cols_t,
    'Gain'   : model_timing.feature_importance(importance_type='gain')
}).sort_values('Gain', ascending=False).head(15)

plt.figure(figsize=(10, 6))
plt.title('구매 시점 예측 – 피처 중요도 (상위 15개)')
sns.barplot(x='Gain', y='Feature', data=imp_t, palette='mako')
for i, v in enumerate(imp_t['Gain']):
    plt.text(v, i, f' {v:,.0f}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

## 6. 모델 2 – 공동구매 강화 재구매 예측

기존 피처에 **타이밍 피처** + **copurchase_lift** 를 추가해 재구매 예측 성능을 향상시킵니다.

In [ ]:
unused2 = ['user_id', 'order_id', 'eval_set', 'product_id',
           'reordered', 'order_number', 'aisle_id']
feat_cols2 = [c for c in data.columns
              if c not in unused2 and data[c].dtype != object]

X2 = data[feat_cols2].fillna(0)
y2 = data['reordered'].astype(int)

X_trv2, X_te2, y_trv2, y_te2 = train_test_split(
    X2, y2, test_size=0.1, random_state=42, stratify=y2)
X_tr2, X_va2, y_tr2, y_va2 = train_test_split(
    X_trv2, y_trv2, test_size=2/9, random_state=42, stratify=y_trv2)

total2 = len(y2)
print('[데이터 분리 결과 (7:2:1)]')
print(f'  학습 : {len(y_tr2):>10,}건  ({len(y_tr2)/total2:.1%})')
print(f'  검증 : {len(y_va2):>10,}건  ({len(y_va2)/total2:.1%})')
print(f'  테스트: {len(y_te2):>10,}건  ({len(y_te2)/total2:.1%})')
print(f'\n사용 피처 수: {len(feat_cols2)}')
print(f'피처 목록: {feat_cols2}')

In [ ]:
scale_w2 = round((y_tr2 == 0).sum() / (y_tr2 == 1).sum(), 2)
dtrain2  = lgb.Dataset(X_tr2, label=y_tr2)
dval2    = lgb.Dataset(X_va2, label=y_va2, reference=dtrain2)

params2 = {
    'objective'        : 'binary',
    'metric'           : 'auc',
    'boosting_type'    : 'gbdt',
    'scale_pos_weight' : scale_w2,
    'learning_rate'    : 0.05,
    'num_leaves'       : 63,
    'feature_fraction' : 0.8,
    'bagging_fraction' : 0.8,
    'bagging_freq'     : 5,
    'min_data_in_leaf' : 100,
    'seed'             : 42,
    'verbose'          : -1
}

model2 = lgb.train(
    params2, dtrain2, num_boost_round=1000,
    valid_sets=[dtrain2, dval2], valid_names=['train', 'valid'],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)]
)
print(f'\n최적 반복 횟수: {model2.best_iteration}')
print(f'최고 검증 AUC : {model2.best_score["valid"]["auc"]:.6f}')

In [ ]:
from sklearn.metrics import f1_score, confusion_matrix, classification_report, accuracy_score

test_probs2 = model2.predict(X_te2)
best_f1_2, best_t2 = 0, 0.5

print(f"{'임계값':<10} | {'F1-Score':<10}")
print('-' * 25)
for t in np.arange(0.1, 0.91, 0.05):
    preds = (test_probs2 >= t).astype(int)
    f1 = f1_score(y_te2, preds)
    print(f'{t:<10.2f} | {f1:<10.4f}')
    if f1 > best_f1_2:
        best_f1_2, best_t2 = f1, t
print('-' * 25)
print(f'최적 임계값: {best_t2:.2f}  |  최고 F1-Score: {best_f1_2:.4f}')

In [ ]:
final_preds2 = (test_probs2 >= best_t2).astype(int)
tn, fp, fn, tp = confusion_matrix(y_te2, final_preds2).ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
precision   = tp / (tp + fp)
npv         = tn / (tn + fn)
prevalence  = (tp + fn) / (tp + tn + fp + fn)
det_rate    = tp / (tp + tn + fp + fn)
det_prev    = (tp + fp) / (tp + tn + fp + fn)
bal_acc     = (sensitivity + specificity) / 2

print('=' * 60)
print(f'  차별화 모델 최종 성능 리포트 (임계값: {best_t2:.2f})')
print('=' * 60)
print(f'  Accuracy             : {accuracy_score(y_te2, final_preds2):.4f}')
print(f'  F1-Score             : {best_f1_2:.4f}')
print()
print(f'  Sensitivity (Recall) : {sensitivity:.5f}  <- 재구매자 중 맞춘 비율')
print(f'  Specificity          : {specificity:.5f}  <- 미구매자 중 맞춘 비율')
print(f'  Pos Pred Value (PPV) : {precision:.5f}  <- 산다 예측 중 실제 구매 비율')
print(f'  Neg Pred Value (NPV) : {npv:.5f}')
print(f'  Prevalence           : {prevalence:.5f}')
print(f'  Detection Rate       : {det_rate:.5f}')
print(f'  Detection Prevalence : {det_prev:.5f}')
print(f'  Balanced Accuracy    : {bal_acc:.5f}')
print('-' * 60)
print()
print('[기본 분류 리포트]')
print(classification_report(y_te2, final_preds2, target_names=['미구매(0)', '재구매(1)']))

## 7. 피처 중요도 및 차별화 피처 기여도 분석

In [ ]:
import matplotlib.patches as mpatches

new_features = ['up_avg_interval', 'up_std_interval', 'days_since_last_buy',
                'timing_ratio', 'hazard_proxy', 'up_buy_count', 'copurchase_lift']

imp2 = pd.DataFrame({
    'Feature': feat_cols2,
    'Gain'   : model2.feature_importance(importance_type='gain'),
    'Split'  : model2.feature_importance(importance_type='split'),
}).sort_values('Gain', ascending=False).reset_index(drop=True)
imp2['is_new'] = imp2['Feature'].isin(new_features)

print('[피처별 Gain (재구매 예측 기여도 내림차순)]')
print(imp2.to_string(index=False))

print('\n[차별화 신규 피처 중요도]')
print(imp2[imp2['is_new']].to_string(index=False))

colors = imp2['is_new'].map({True: '#e74c3c', False: '#3498db'}).tolist()
fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(imp2['Feature'], imp2['Gain'], color=colors)
ax.set_xlabel('Importance (Total Gain)')
ax.set_title('피처 중요도 (빨간색=차별화 신규 피처, 파란색=기존 피처)')
ax.invert_yaxis()
legend_handles = [
    mpatches.Patch(color='#e74c3c', label='신규 피처 (타이밍 + 공동구매 Lift)'),
    mpatches.Patch(color='#3498db', label='기존 피처')
]
ax.legend(handles=legend_handles, loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# 상위 15개 공동 구매 Aisle 조합 (lift 기준 대칭 평균)
if 'aisle_name' not in vars():
    aisle_name = dict(zip(aisles['aisle_id'], aisles['aisle']))

sym_pairs = []
for i in range(len(all_aisle_ids)):
    for j in range(i + 1, len(all_aisle_ids)):
        s = float((lift_matrix[i, j] + lift_matrix[j, i]) / 2)
        if s > 1.0:
            sym_pairs.append({
                'aisle_a': aisle_name.get(all_aisle_ids[i], str(all_aisle_ids[i])),
                'aisle_b': aisle_name.get(all_aisle_ids[j], str(all_aisle_ids[j])),
                'lift'   : round(s, 3)
            })

sym_df   = pd.DataFrame(sym_pairs).sort_values('lift', ascending=False).head(15)
y_labels = [f"{r['aisle_a'][:22]}\n+ {r['aisle_b'][:22]}" for _, r in sym_df.iterrows()]

plt.figure(figsize=(12, 8))
plt.title('상위 15개 공동 구매 Aisle 조합 (Lift 기준, 높을수록 강한 연관)')
plt.barh(range(len(sym_df)), sym_df['lift'].values, color='#2ecc71')
plt.yticks(range(len(sym_df)), y_labels, fontsize=8)
plt.xlabel('Co-purchase Lift Score')
plt.axvline(x=1.0, color='gray', linestyle='--', alpha=0.7, label='Lift=1 (무관)')
plt.legend()
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print('\n[상위 15개 공동 구매 패턴 (Lift 점수)]')
print(sym_df.to_string(index=False))

In [ ]:
# hazard_proxy 구간별 실제 재구매율 검증
# hazard_proxy가 높을수록 평균보다 오래 경과 → 실제 재구매율이 높아야 피처가 유효
analysis = data[data['hazard_proxy'] > 0].copy()
analysis['hazard_bin'] = pd.cut(
    analysis['hazard_proxy'].clip(upper=5),
    bins=[0, 0.5, 1.0, 1.5, 2.0, 5.0],
    labels=['0-0.5 (빠름)', '0.5-1.0 (적당)', '1.0-1.5 (약간 overdue)',
            '1.5-2.0 (overdue)', '>2.0 (많이 overdue)']
)

hazard_analysis = (
    analysis.groupby('hazard_bin', observed=True)['reordered']
    .agg(['mean', 'count'])
    .rename(columns={'mean': '실제_재구매율', 'count': '샘플수'})
    .reset_index()
)

print('[hazard_proxy 구간별 실제 재구매율]')
print('(값이 높을수록 실제 재구매율이 높아야 피처가 유효함)')
print(hazard_analysis.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(len(hazard_analysis)), hazard_analysis['실제_재구매율'], color='#9b59b6')
axes[0].set_xticks(range(len(hazard_analysis)))
axes[0].set_xticklabels(hazard_analysis['hazard_bin'], rotation=20, ha='right', fontsize=8)
axes[0].set_ylabel('실제 재구매율')
axes[0].set_title('hazard_proxy 구간별 실제 재구매율')

# copurchase_lift 분위별 재구매율
analysis['lift_quantile'] = pd.qcut(
    analysis['copurchase_lift'], q=5,
    labels=['Q1 (낮음)', 'Q2', 'Q3', 'Q4', 'Q5 (높음)'],
    duplicates='drop'
)
lift_analysis = (
    analysis.groupby('lift_quantile', observed=True)['reordered']
    .agg(['mean', 'count'])
    .rename(columns={'mean': '실제_재구매율', 'count': '샘플수'})
    .reset_index()
)
axes[1].bar(range(len(lift_analysis)), lift_analysis['실제_재구매율'], color='#e67e22')
axes[1].set_xticks(range(len(lift_analysis)))
axes[1].set_xticklabels(lift_analysis['lift_quantile'], rotation=15, ha='right', fontsize=9)
axes[1].set_ylabel('실제 재구매율')
axes[1].set_title('copurchase_lift 분위별 실제 재구매율')

plt.tight_layout()
plt.show()

print('\n[copurchase_lift 분위별 실제 재구매율]')
print(lift_analysis.to_string(index=False))